In [7]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())

True
1


In [8]:
import torch

#from datasets import load_dataset,Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig
#from huggingface_hub import hf_xet
from peft import LoraConfig, get_peft_config, TaskType, get_peft_model

In [9]:

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


CUDA available: True
GPU name: NVIDIA GeForce RTX 4060 Laptop GPU


In [10]:
from transformers import AutoModelForImageTextToText, AutoProcessor

model_id = "google/medgemma-4b-it"
save_path = "D:/mini/models/medgemma-8bit"

print("Loading MedGemma in 8-bit...")

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    load_in_8bit=True,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    cache_dir="D:/mini/huggingface_cache"
)

processor = AutoProcessor.from_pretrained(
    model_id,
    cache_dir="D:/mini/huggingface_cache"
)

print("Saving quantized model to:", save_path)

model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print("✅ Quantized 8-bit model saved!")


Loading MedGemma in 8-bit...


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
d:\mini\train\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\mini\huggingface_cache\models--google--medgemma-4b-it. Caching fil

Saving quantized model to: D:/mini/models/medgemma-8bit
✅ Quantized 8-bit model saved!


In [ ]:
import gc
gc.collect()

import torch
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


: 

In [2]:
import transformers
from transformers import AutoModelForCausalLM , AutoTokenizer , pipeline
import torch

model_id = "ContactDoctor/Bio-Medical-Llama-3-2-1B-CoT-012025"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    cache_dir="d:/mini/huggingface_model"
)

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True,
    cache_dir="d:/mini/huggingface_model"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
)

messages = [
    {"role": "system", "content": "You are a biomedical expert."},
    {"role": "user", "content": "List causes of chest pain in one line."},
]

prompt = pipe.tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

result = pipe(prompt, max_new_tokens=100)
print(result[0]["generated_text"])


d:\mini\train\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\mini\huggingface_model\models--ContactDoctor--Bio-Medical-Llama-3-2-1B-CoT-012025. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 08 Nov 2025

You are a biomedical expert.<|eot_id|><|start_header_id|>user<|end_header_id|>

List causes of chest pain in one line.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Here are some common causes of chest pain that I can list in one line:

- Heart attack or myocardial infarction
- Aneurysm rupture
- Pneumonia
- Pneumothorax
- Pulmonary embolism
- Cardiac arrhythmias
- Cor, aortic aneurysm
- Angina pectoris
- Atrial fibrillation
- Cardiac tamponade
- Cardiogenic shock
- Pneumonia


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_id = "HiTZ/Medical-mT5-large"

tokenizer = AutoTokenizer.from_pretrained(model_id,cache_dir="d:\\mini\\huggingface_model")
model = AutoModelForSeq2SeqLM.from_pretrained(model_id,
                                              #cache_dir="d:\\mini\\huggingface_model"
                                              )

# Example prompt
input_text = "Summarize the following medical case: A 55-year-old man has chest pain and shortness of breath."

# Tokenize
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

# Run generation
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=100, num_beams=4, early_stopping=True)

# Decode
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Result:", result)


d:\mini\train\lib\site-packages\huggingface_hub\file_download.py:798: UserWarning: Not enough free disk space to download the file. The expected file size is: 4918.39 MB. The target location C:\Users\tanishq\.cache\huggingface\hub\models--HiTZ--Medical-mT5-large\blobs only has 1793.71 MB free disk space.
  warnings.warn(


OSError: Can't load the model for 'HiTZ/Medical-mT5-large'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'HiTZ/Medical-mT5-large' is the correct path to a directory containing a file named pytorch_model.bin, tf_model.h5, model.ckpt or flax_model.msgpack.